# Train Stockout Risk Model

Trains an XGBoost classifier on data from `scripts/generate_training_data.py`.
Outputs `app/data/stockout_model.pkl` (a dict with `model` and `feature_cols`).

In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

DATA_PATH = Path('../app/data/training_data.parquet')
MODEL_PATH = Path('../app/data/stockout_model.pkl')

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(df):,} rows')
print(df['stockout_risk'].value_counts())

Loaded 23,800 rows
stockout_risk
1    14886
0     8914
Name: count, dtype: int64


In [3]:
SIZE_MAP = {'small': 0, 'medium': 1, 'large': 2}
df['store_size'] = df['store_size'].map(SIZE_MAP)

FEATURE_COLS = [
    'demand_multiplier',
    'nearby_stores_closed',
    'days_supply_disrupted',
    'weather_severity',
    'store_size',
]

X = df[FEATURE_COLS].to_numpy(dtype=float)
y = df['stockout_risk'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

Train: 19,040  Test: 4,760


In [4]:
clf = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('Training complete')

Training complete


In [5]:
proba = clf.predict_proba(X_test)[:, 1]
preds = clf.predict(X_test)

print(f'AUC-ROC : {roc_auc_score(y_test, proba):.4f}')
print()
print(classification_report(y_test, preds, target_names=['no stockout', 'stockout']))

AUC-ROC : 0.9982

              precision    recall  f1-score   support

 no stockout       0.98      0.96      0.97      1783
    stockout       0.98      0.99      0.98      2977

    accuracy                           0.98      4760
   macro avg       0.98      0.98      0.98      4760
weighted avg       0.98      0.98      0.98      4760



In [6]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURE_COLS,
).sort_values(ascending=False)

print('Mean absolute SHAP value per feature:')
print(mean_abs_shap.round(4).to_string())

Mean absolute SHAP value per feature:
nearby_stores_closed     2.5638
demand_multiplier        2.3051
days_supply_disrupted    1.6462
store_size               0.9676
weather_severity         0.6846


In [7]:
artifact = {'model': clf, 'feature_cols': FEATURE_COLS}
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)
print(f'Saved → {MODEL_PATH.resolve()}')

Saved → /Users/courtneyquinn/boston-logistics-sim/backend/app/data/stockout_model.pkl
